In [1]:
import pandas as pd

df = pd.read_json('/home/bartoszworklinux/std/bcmp-networks/logs/vcXXWKIftD.jsonl', lines=True)


In [11]:
df

,source,id,starttime,duration,config_path,ts,action,request_id,type
0,INIT,vcXXWKIftD,2024-11-13T19:06:48.964098,30.0,configs/networks/network.json,NaN,NaN,NaN,NaN
1,WS 1,NaN,NaN,NaN,NaN,NaN,received,118.0,Krytyczny
2,WS 1,NaN,NaN,NaN,NaN,NaN,queued,118.0,Krytyczny
3,GN 2,NaN,NaN,NaN,NaN,NaN,generated,118.0,Krytyczny
4,WS 1,NaN,NaN,NaN,NaN,NaN,started,118.0,Krytyczny
...,...,...,...,...,...,...,...,...,...
126,WS 1,NaN,NaN,NaN,NaN,NaN,processed,140.0,Krytyczny
127,WS 1,NaN,NaN,NaN,NaN,NaN,started,141.0,Krytyczny
128,WS 1,NaN,NaN,NaN,NaN,NaN,received,143.0,Krytyczny
129,WS 1,NaN,NaN,NaN,NaN,NaN,queued,143.0,Krytyczny


In [3]:
# Convert the timestamp 'ts' column to a standard datetime.timedelta format for easy time calculations
import numpy as np
import datetime as dt
def avg_requests_per_class(df):
# Convert 'ts' to timedelta
    def parse_timestamp(ts):
        try:
            minutes, seconds, milliseconds = map(int, ts.split(':'))
            return dt.timedelta(minutes=minutes, seconds=seconds, milliseconds=milliseconds)
        except:
            return np.nan

    
    df['ts'] = df['ts'].apply(parse_timestamp)

    # Filter relevant rows for each calculation by actions
    # Extracting queueing and processing stages
    queued_df = df[df['action'] == 'queued']
    started_df = df[df['action'] == 'started']
    processed_df = df[df['action'] == 'processed']

    # Calculating average number of requests by class and server
    avg_requests_per_class_server = df.groupby(['source', 'type'])['request_id'].nunique().groupby(level=0).mean()


    # Combining results into a single DataFrame for presentation
    results_df = pd.DataFrame({
        'avg_requests_per_class': avg_requests_per_class_server,
    }).reset_index()


    return results_df

avg_requests_per_class(df)
    

,source,avg_requests_per_class
0,GN 2,26.0
1,WS 1,26.0


In [10]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_avg_time_request_server_with_type(data_subset, server_name):
    # Konwersja 'ts' na timedelta
    data_subset['ts'] = pd.to_timedelta(data_subset['ts'])
    print(data_subset)
    data_subset = data_subset.sort_values(by=['request_id', 'ts'])

    processing_intervals = []
    new_request_id = None

    type_color_mapping = {
        "type1": "green",
        "type2": "yellow",
        "type3": "red",
    }

    for request_id, group in data_subset.groupby('request_id'):
        group = group.reset_index(drop=True)
        last_type = None
        for i, row in group.iterrows():
            if row['action'] == 'received' or (last_type and row.get('new_type') != last_type):
                current_type = row.get('new_type') if row.get('new_type') else 'initial'
                new_request_id = f"{request_id}_{i}_{current_type}"
                
            if row['action'] == 'received':
                start_time = row['ts']
                last_type = row['type']
            
            elif row['action'] in ['forwarded', 'rejected']:
                end_time = row['ts']
                processing_time = (end_time - start_time).total_seconds()

                if processing_time > 0:
                    processing_intervals.append({
                        'request_id': new_request_id, 
                        'processing_time': processing_time, 
                        'type': last_type
                    })
                    
                last_type = row.get('new_type', 'initial')

    processing_intervals_df = pd.DataFrame(processing_intervals)
    if not processing_intervals_df.empty:
        processing_intervals_df['new_type'] = processing_intervals_df['type'].map(type_color_mapping)
        average_processing_time_df = processing_intervals_df.groupby(['request_id', 'new_type'])['processing_time'].mean().reset_index()
        
        # Tworzenie listy kolorów na podstawie mappingu
        colors = average_processing_time_df['new_type'].map(type_color_mapping)

        plt.figure(figsize=(10, 6))
        plt.bar(average_processing_time_df['request_id'], average_processing_time_df['processing_time'], color=colors)
        plt.xlabel('Request ID (with Type)')
        plt.ylabel('Average Processing Time (seconds)')
        plt.title(f'Average Processing Time for each Request ID with Type ||| {server_name}')
        
        legend_labels = {
            "green": "Symulant",
            "yellow": "Stabilny",
            "red": "Krytyczny"
        }
        handles = [plt.Line2D([0], [0], marker='o', color=color, label=label, markersize=10, linestyle='None')
                   for color, label in legend_labels.items()]
        plt.legend(handles=handles, title="Typy")
        plt.xticks([], [])  
        plt.show()
    else:
        print(f"No valid processing times for source subset.")

# Sprawdzenie unikalnych źródeł w 'source' i wywołanie funkcji dla każdego
# for source in df['source'].unique():
#     if "WS" in source:
#         data_subset = df[df['source'] == source]
#         print(f"Processing data for source: {source}")
#         plot_avg_time_request_server_with_type(data_subset, server_name=source)
source = 'WS 1'
data_subset = df[df['source'] == source]
plot_avg_time_request_server_with_type(data_subset, server_name=source)


    source   id starttime  duration config_path  ts     action  request_id  \
1     WS 1  NaN       NaN       NaN         NaN NaT   received       118.0   
2     WS 1  NaN       NaN       NaN         NaN NaT     queued       118.0   
4     WS 1  NaN       NaN       NaN         NaN NaT    started       118.0   
5     WS 1  NaN       NaN       NaN         NaN NaT   received       119.0   
6     WS 1  NaN       NaN       NaN         NaN NaT     queued       119.0   
..     ...  ...       ...       ...         ...  ..        ...         ...   
125   WS 1  NaN       NaN       NaN         NaN NaT   finished       140.0   
126   WS 1  NaN       NaN       NaN         NaN NaT  processed       140.0   
127   WS 1  NaN       NaN       NaN         NaN NaT    started       141.0   
128   WS 1  NaN       NaN       NaN         NaN NaT   received       143.0   
129   WS 1  NaN       NaN       NaN         NaN NaT     queued       143.0   

          type  
1    Krytyczny  
2    Krytyczny  
4    Krytycz

/tmp/ipykernel_33424/8161809.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_subset['ts'] = pd.to_timedelta(data_subset['ts'])
